In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from pathlib import Path
import glob

# --- Configuration ---
RESULTS_DIR = "/data/cpanourg/2-hdvc/results/relerr/"
OUTPUT_DIR = Path("/home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/second_hp_test")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)  # Create directory if it doesn't exist

# --- Style (paper-quality minimalist) ---
plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.size": 8,
    "font.family": "serif",
    "axes.labelsize": 8,
    "axes.titlesize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# --- Find all CSV files ---
csv_files = glob.glob(f"{RESULTS_DIR}/deep_*_adc_vs_exact_eval.csv")
methods = {}
for csv_file in csv_files:
    method_name = Path(csv_file).stem.replace("deep_", "").replace("_adc_vs_exact_eval", "")
    methods[method_name] = csv_file

print(f"Found methods: {list(methods.keys())}")

# --- Define x-axis variables for each method ---
x_axis_configs = {
    "PQ": ["train_size", "n_subquantizers", "nbits", "bits_per_vector"],
    "OPQ": ["train_size", "n_subquantizers", "nbits", "bits_per_vector"],
    "LSQpp": ["train_size", "n_subquantizers", "nbits", "bits_per_vector"],
    "RaBitQ": ["train_size", "bits_per_dim", "bits_per_vector"],
    "VAQ": ["train_size", "n_subquantizers", "bits_per_vector", "min_bits", "max_bits"],
}

# --- X-axis label mapping ---
x_labels = {
    "train_size": "Training size",
    "n_subquantizers": "Number of subquantizers",
    "nbits": "Bits per subspace",
    "bits_per_vector": "Bits per vector",
    "bits_per_dim": "Bits per dimension",
    "min_bits": "Minimum bits per subspace",
    "max_bits": "Maximum bits per subspace",
}

# --- Function to perform significance test ---
def perform_significance_test(x, y):
    """
    Perform Pearson correlation test and return p-value and sample size.
    
    Pearson correlation is valid for testing linear relationships between continuous variables.
    It tests H0: correlation = 0 (no linear relationship) vs H1: correlation != 0.
    Returns p-value < 0.05 indicates significant linear relationship.
    """
    # Remove NaN values
    mask = ~(np.isnan(x) | np.isnan(y))
    x_clean = np.array(x)[mask]
    y_clean = np.array(y)[mask]
    
    if len(x_clean) < 3:
        return None, None
    
    # Check if either variable is constant (correlation undefined)
    if np.std(x_clean) == 0 or np.std(y_clean) == 0:
        return None, None
    
    # Pearson correlation test
    try:
        corr, p_value = stats.pearsonr(x_clean, y_clean)
        return p_value, len(x_clean)
    except:
        return None, None

# --- Generate plots for each method ---
for method_name, csv_file in methods.items():
    print(f"\nProcessing {method_name}...")
    df = pd.read_csv(csv_file)
    
    # Get x-axis variables for this method
    method_x_vars = x_axis_configs.get(method_name, ["train_size", "bits_per_vector"])
    
    # Filter to only variables that exist in the dataframe
    available_x_vars = [x for x in method_x_vars if x in df.columns]
    
    for x_var in available_x_vars:
        # Skip if no valid data
        if df[x_var].isna().all():
            continue
        
        # Remove rows with NaN in x or y variables
        df_clean = df[[x_var, "rel_error_mean", "rel_error_std"]].dropna()
        
        if len(df_clean) < 2:
            continue
        
        # --- Plot 1: Mean Relative Error ---
        fig, ax = plt.subplots(figsize=(3.3, 2.2))
        
        # Scatter plot
        ax.scatter(
            df_clean[x_var], df_clean["rel_error_mean"],
            s=20, color="0.6", alpha=0.7, linewidths=0.3, edgecolors="0.4"
        )
        
        # Highlight best configuration
        best_idx = df_clean["rel_error_mean"].idxmin()
        best = df_clean.loc[best_idx]
        ax.scatter(
            best[x_var], best["rel_error_mean"],
            s=70, marker="*", color="black", edgecolors="black", zorder=3
        )
        
        # Perform significance test
        p_val, n_samples = perform_significance_test(df_clean[x_var], df_clean["rel_error_mean"])
        
        # Add significance test result to plot (top-right corner to avoid overlapping points)
        if p_val is not None:
            sig_text = "significant" if p_val < 0.05 else "not significant"
            test_text = f"p={p_val:.3e} ({sig_text})\nn={n_samples}"
            ax.text(0.98, 0.98, test_text, transform=ax.transAxes,
                   fontsize=7, verticalalignment='top', horizontalalignment='right',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        # Labels and formatting
        ax.set_xlabel(x_labels.get(x_var, x_var))
        ax.set_ylabel("Mean relative error")
        ax.grid(True, linestyle=":", linewidth=0.6, alpha=0.5)
        
        # Format x-axis
        if x_var == "train_size":
            ax.ticklabel_format(style='sci', axis='x', scilimits=(4,6))
        elif x_var in ["nbits", "n_subquantizers", "bits_per_dim", "min_bits", "max_bits"]:
            ax.set_xticks(sorted(df_clean[x_var].unique()))
        
        plt.tight_layout(pad=0.3)
        
        # Save figure
        filename = f"{x_var}_{method_name}_mean_rel_error.pdf"
        filepath = OUTPUT_DIR / filename
        plt.savefig(filepath, bbox_inches="tight")
        print(f"  Saved: {filepath}")
        plt.close()
        
        # --- Plot 2: Standard Deviation of Relative Error ---
        fig, ax = plt.subplots(figsize=(3.3, 2.2))
        
        # Scatter plot
        ax.scatter(
            df_clean[x_var], df_clean["rel_error_std"],
            s=20, color="0.6", alpha=0.7, linewidths=0.3, edgecolors="0.4"
        )
        
        # Highlight configuration with minimum std (associated with best mean)
        best_idx = df_clean["rel_error_mean"].idxmin()
        best = df_clean.loc[best_idx]
        ax.scatter(
            best[x_var], best["rel_error_std"],
            s=70, marker="*", color="black", edgecolors="black", zorder=3
        )
        
        # Perform significance test
        p_val, n_samples = perform_significance_test(df_clean[x_var], df_clean["rel_error_std"])
        
        # Add significance test result to plot (top-right corner to avoid overlapping points)
        if p_val is not None:
            sig_text = "significant" if p_val < 0.05 else "not significant"
            test_text = f"p={p_val:.3e} ({sig_text})\nn={n_samples}"
            ax.text(0.98, 0.98, test_text, transform=ax.transAxes,
                   fontsize=7, verticalalignment='top', horizontalalignment='right',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        # Labels and formatting
        ax.set_xlabel(x_labels.get(x_var, x_var))
        ax.set_ylabel("Std dev of relative error")
        ax.grid(True, linestyle=":", linewidth=0.6, alpha=0.5)
        
        # Format x-axis
        if x_var == "train_size":
            ax.ticklabel_format(style='sci', axis='x', scilimits=(4,6))
        elif x_var in ["nbits", "n_subquantizers", "bits_per_dim", "min_bits", "max_bits"]:
            ax.set_xticks(sorted(df_clean[x_var].unique()))
        
        plt.tight_layout(pad=0.3)
        
        # Save figure
        filename = f"{x_var}_{method_name}_std_rel_error.pdf"
        filepath = OUTPUT_DIR / filename
        plt.savefig(filepath, bbox_inches="tight")
        print(f"  Saved: {filepath}")
        plt.close()

print("\n✅ All plots generated successfully!")


Found methods: ['PQ']

Processing PQ...
  Saved: /home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/second_hp_test/train_size_PQ_mean_rel_error.pdf
  Saved: /home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/second_hp_test/train_size_PQ_std_rel_error.pdf
  Saved: /home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/second_hp_test/n_subquantizers_PQ_mean_rel_error.pdf
  Saved: /home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/second_hp_test/n_subquantizers_PQ_std_rel_error.pdf
  Saved: /home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/second_hp_test/nbits_PQ_mean_rel_error.pdf
  Saved: /home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/second_hp_test/nbits_PQ_std_rel_error.pdf
  Saved: /home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/second_hp_test/bits_per_vector_PQ_mean_rel_error.pdf
  Saved: /home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/second_hp_test/bits_per_vector

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import glob
from scipy import stats

# --- Configuration ---
RESULTS_DIR = "/data/cpanourg/2-hdvc/results/relerr/hp_tests/first_hp_test/"
OUTPUT_DIR = Path("/home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/first_hp_test/combined")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Style (paper-quality minimalist) ---
plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.size": 8,
    "font.family": "serif",
    "axes.labelsize": 8,
    "axes.titlesize": 9,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# --- Find all CSV files ---
csv_files = glob.glob(f"{RESULTS_DIR}/deep_*_adc_vs_exact_eval.csv")
methods_data = {}
for csv_file in csv_files:
    method_name = Path(csv_file).stem.replace("deep_", "").replace("_adc_vs_exact_eval", "")
    methods_data[method_name] = pd.read_csv(csv_file)

print(f"Found methods: {list(methods_data.keys())}")

# --- Define x-axis variables for each method ---
x_axis_configs = {
    "PQ": ["train_size", "n_subquantizers", "nbits", "bits_per_vector"],
    "OPQ": ["train_size", "n_subquantizers", "nbits", "bits_per_vector"],
    "LSQpp": ["train_size", "n_subquantizers", "nbits", "bits_per_vector"],
    "RaBitQ": ["train_size", "bits_per_dim", "bits_per_vector"],
    "VAQ": ["train_size", "n_subquantizers", "bits_per_vector", "min_bits", "max_bits"],
}

# --- X-axis label mapping ---
x_labels = {
    "train_size": "Training size",
    "n_subquantizers": "Number of subquantizers",
    "nbits": "Bits per subspace",
    "bits_per_vector": "Bits per vector",
    "bits_per_dim": "Bits per dimension",
    "min_bits": "Minimum bits per subspace",
    "max_bits": "Maximum bits per subspace",
}

# --- Function to perform significance test ---
def perform_significance_test(x, y):
    """
    Perform Pearson correlation test and return p-value and sample size.
    
    Pearson correlation is valid for testing linear relationships between continuous variables.
    It tests H0: correlation = 0 (no linear relationship) vs H1: correlation != 0.
    Returns p-value < 0.05 indicates significant linear relationship.
    """
    # Remove NaN values
    mask = ~(np.isnan(x) | np.isnan(y))
    x_clean = np.array(x)[mask]
    y_clean = np.array(y)[mask]
    
    if len(x_clean) < 3:
        return None, None
    
    # Check if either variable is constant (correlation undefined)
    if np.std(x_clean) == 0 or np.std(y_clean) == 0:
        return None, None
    
    # Pearson correlation test
    try:
        corr, p_value = stats.pearsonr(x_clean, y_clean)
        return p_value, len(x_clean)
    except:
        return None, None

# --- Find hyperparameters that exist in multiple methods ---
# Collect all hyperparameters and which methods have them
hyperparam_methods = {}
for method_name, df in methods_data.items():
    method_x_vars = x_axis_configs.get(method_name, [])
    for x_var in method_x_vars:
        if x_var in df.columns and not df[x_var].isna().all():
            if x_var not in hyperparam_methods:
                hyperparam_methods[x_var] = []
            hyperparam_methods[x_var].append(method_name)

# Only keep hyperparameters that exist in at least 2 methods
hyperparam_methods = {k: v for k, v in hyperparam_methods.items() if len(v) >= 2}

print(f"\nHyperparameters with multiple methods: {list(hyperparam_methods.keys())}")

# --- Generate combined figures for each hyperparameter ---
for x_var, method_list in hyperparam_methods.items():
    print(f"\nProcessing {x_var} for methods: {method_list}")
    
    # Collect all data for this hyperparameter
    all_data = {}
    x_min, x_max = np.inf, -np.inf
    y_mean_min, y_mean_max = np.inf, -np.inf
    y_std_min, y_std_max = np.inf, -np.inf
    
    for method_name in method_list:
        df = methods_data[method_name]
        df_clean = df[[x_var, "rel_error_mean", "rel_error_std"]].dropna()
        
        if len(df_clean) < 2:
            continue
        
        all_data[method_name] = df_clean
        
        # Update global ranges
        x_min = min(x_min, df_clean[x_var].min())
        x_max = max(x_max, df_clean[x_var].max())
        y_mean_min = min(y_mean_min, df_clean["rel_error_mean"].min())
        y_mean_max = max(y_mean_max, df_clean["rel_error_mean"].max())
        y_std_min = min(y_std_min, df_clean["rel_error_std"].min())
        y_std_max = max(y_std_max, df_clean["rel_error_std"].max())
    
    if len(all_data) < 2:
        continue
    
    # Add small padding to ranges
    x_range = x_max - x_min
    y_mean_range = y_mean_max - y_mean_min
    y_std_range = y_std_max - y_std_min
    
    x_min -= x_range * 0.05
    x_max += x_range * 0.05
    y_mean_min -= y_mean_range * 0.05
    y_mean_max += y_mean_range * 0.05
    y_std_min -= y_std_range * 0.05
    y_std_max += y_std_range * 0.05
    
    # Create figure with 2 rows (mean, std) and N columns (one per method)
    n_methods = len(all_data)
    fig, axes = plt.subplots(2, n_methods, figsize=(3.3 * n_methods, 4.4))
    
    # If only one method, axes will be 1D, convert to 2D
    if n_methods == 1:
        axes = axes.reshape(2, 1)
    
    # Plot mean relative error (row 0)
    for col_idx, (method_name, df_clean) in enumerate(all_data.items()):
        ax = axes[0, col_idx]
        
        # Scatter plot
        ax.scatter(
            df_clean[x_var], df_clean["rel_error_mean"],
            s=20, color="0.6", alpha=0.7, linewidths=0.3, edgecolors="0.4"
        )
        
        # Highlight best configuration
        best_idx = df_clean["rel_error_mean"].idxmin()
        best = df_clean.loc[best_idx]
        ax.scatter(
            best[x_var], best["rel_error_mean"],
            s=70, marker="*", color="black", edgecolors="black", zorder=3
        )
        
        # Set same limits for all subplots
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_mean_min, y_mean_max)
        
        # Perform significance test
        p_val, n_samples = perform_significance_test(df_clean[x_var], df_clean["rel_error_mean"])
        
        # Add significance test result to plot (top-right corner)
        if p_val is not None:
            sig_text = "significant" if p_val < 0.05 else "not significant"
            test_text = f"p={p_val:.3e} ({sig_text})\nn={n_samples}"
            ax.text(0.98, 0.98, test_text, transform=ax.transAxes,
                   fontsize=6, verticalalignment='top', horizontalalignment='right',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        # Labels and formatting
        if col_idx == 0:
            ax.set_ylabel("Mean relative error")
        ax.set_title(method_name, fontsize=9)
        ax.grid(True, linestyle=":", linewidth=0.6, alpha=0.5)
        
        # Format x-axis (only show label on bottom row)
        if x_var == "train_size":
            ax.ticklabel_format(style='sci', axis='x', scilimits=(4,6))
        elif x_var in ["nbits", "n_subquantizers", "bits_per_dim", "min_bits", "max_bits"]:
            unique_vals = sorted(df_clean[x_var].unique())
            ax.set_xticks(unique_vals)
    
    # Plot std relative error (row 1)
    for col_idx, (method_name, df_clean) in enumerate(all_data.items()):
        ax = axes[1, col_idx]
        
        # Scatter plot
        ax.scatter(
            df_clean[x_var], df_clean["rel_error_std"],
            s=20, color="0.6", alpha=0.7, linewidths=0.3, edgecolors="0.4"
        )
        
        # Highlight configuration with minimum std (associated with best mean)
        best_idx = df_clean["rel_error_mean"].idxmin()
        best = df_clean.loc[best_idx]
        ax.scatter(
            best[x_var], best["rel_error_std"],
            s=70, marker="*", color="black", edgecolors="black", zorder=3
        )
        
        # Set same limits for all subplots
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_std_min, y_std_max)
        
        # Perform significance test
        p_val, n_samples = perform_significance_test(df_clean[x_var], df_clean["rel_error_std"])
        
        # Add significance test result to plot (top-right corner)
        if p_val is not None:
            sig_text = "significant" if p_val < 0.05 else "not significant"
            test_text = f"p={p_val:.3e} ({sig_text})\nn={n_samples}"
            ax.text(0.98, 0.98, test_text, transform=ax.transAxes,
                   fontsize=6, verticalalignment='top', horizontalalignment='right',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        # Labels and formatting
        ax.set_xlabel(x_labels.get(x_var, x_var))
        if col_idx == 0:
            ax.set_ylabel("Std dev of relative error")
        ax.grid(True, linestyle=":", linewidth=0.6, alpha=0.5)
        
        # Format x-axis
        if x_var == "train_size":
            ax.ticklabel_format(style='sci', axis='x', scilimits=(4,6))
        elif x_var in ["nbits", "n_subquantizers", "bits_per_dim", "min_bits", "max_bits"]:
            unique_vals = sorted(df_clean[x_var].unique())
            ax.set_xticks(unique_vals)
    
    plt.tight_layout(pad=0.5)
    
    # Save figure
    filename = f"{x_var}_combined_mean_std.pdf"
    filepath = OUTPUT_DIR / filename
    plt.savefig(filepath, bbox_inches="tight")
    print(f"  Saved: {filepath}")
    plt.close()

print("\n✅ All combined plots generated successfully!")


Found methods: ['PQ', 'LSQpp', 'OPQ', 'VAQ', 'RaBitQ']

Hyperparameters with multiple methods: ['train_size', 'n_subquantizers', 'nbits', 'bits_per_vector']

Processing train_size for methods: ['PQ', 'LSQpp', 'OPQ', 'VAQ', 'RaBitQ']


  Saved: /home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/first_hp_test/combined/train_size_combined_mean_std.pdf

Processing n_subquantizers for methods: ['PQ', 'LSQpp', 'OPQ', 'VAQ']
  Saved: /home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/first_hp_test/combined/n_subquantizers_combined_mean_std.pdf

Processing nbits for methods: ['PQ', 'LSQpp', 'OPQ']
  Saved: /home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/first_hp_test/combined/nbits_combined_mean_std.pdf

Processing bits_per_vector for methods: ['PQ', 'LSQpp', 'OPQ', 'VAQ', 'RaBitQ']
  Saved: /home/cpanourg/projects/2-hdvc/notebooks/general_results/figures/first_hp_test/combined/bits_per_vector_combined_mean_std.pdf

✅ All combined plots generated successfully!
